Fine-Tuning GPT-2(GPT-2 already knows English,We train it on specific data (reviews / recipes)So it learns a specific style)
1: PRODUCT REVIEW GENERATOR

In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch, math
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling, set_seed
from datasets import Dataset

set_seed(42)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt')

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

baseline = {}

print("=== BEFORE FINE-TUNING ===")
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(p, "→", baseline[p], "\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BEFORE FINE-TUNING ===
This product is → This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most 

I bought this phone and → I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 processor with the 8 

The quality of this item → The

In [4]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments perfectly.",
    "the sound quality of these headphones is incredible.",
    "this smartwatch tracks my steps accurately.",
    "great wireless earbuds with noise cancellation.",
]

dataset = Dataset.from_dict({"text": corpus})

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [5]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.662454
10,1.827990
15,1.082190
20,1.019946
25,0.773634
30,0.596344


TrainOutput(global_step=30, training_loss=1.4937597354253134, metrics={'train_runtime': 156.1236, 'train_samples_per_second': 0.32, 'train_steps_per_second': 0.192, 'total_flos': 3266150400000.0, 'train_loss': 1.4937597354253134, 'epoch': 10.0})

In [6]:
print("\n=== AFTER FINE-TUNING ===")

for p in review_prompts:
    output = generate_text(model, tokenizer, p)
    print(p)
    print("Before:", baseline[p][:100])
    print("After :", output[:100])
    print()


=== AFTER FINE-TUNING ===
This product is
Before: This product is made from high quality, lightweight stainless steel. If you are looking for somethin
After : This product is compatible with all models of the following models - 2014 Honda CRX5i, 2008 Honda CR

I bought this phone and
Before: I bought this phone and I have not used it on a lot of people. I have also not used it on any other 
After : I bought this phone and it performs flawlessly and is a great value for the money. The camera qualit

The quality of this item
Before: The quality of this item in the item description (and if the item is already in stock) will determin
After : The quality of this item is outstanding for the price. The only thing I would say about this item is



2: RECIPE GENERATOR

In [7]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

baseline2 = {}

print("=== BEFORE TRAINING ===")

for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(p, "→", baseline2[p], "\n")

=== BEFORE TRAINING ===
To make butter chicken → To make butter chicken, cook 2 pieces and mix in the butter. When the butter is softened, add the remaining 1/3 cup of chicken fat and stir until it is thickened. Stir in the soy sauce and the chicken. Return to the bowl and add the chicken and chicken broth. Set aside, uncovered. Once the soup is done and the chicken is tender, add the broccoli and vegetables and serve immediately.

4. I am going to use your favorite vegan rice cooker. It 

For pasta carbonara → For pasta carbonara, try this dish:

Pavlovioli. It is made with fresh mozzarella, garlic, basil, oregano and Parmesan. It is really good, right? (And it is delicious!)

This is one of my top 20 best Italian pasta recipes. You can see it here: http://pasta.com/2012/08/12/pavlovioli-pepero/

I'm glad you asked 

To prepare chocolate cake → To prepare chocolate cake for your kids, make sure you're using the right amount of sugar, and keep a little bit of it in your bowl to prevent

In [9]:
recipes = [
    "to make butter chicken marinate chicken in yogurt and spices.",
    "heat oil and cook onions until golden brown.",
    "add tomato puree and cook until thick.",
    "add chicken and cook for 15 minutes.",
    "serve with cream and rice."
]

dataset2 = Dataset.from_dict({"text": recipes})

def tokenize2(example):
    return tokenizer2(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized2 = dataset2.map(tokenize2, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [10]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_strategy="no"
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=tokenized2,
    data_collator=collator2
)

trainer2.train()

Step,Training Loss
5,3.321832
10,1.880296
15,1.113419
20,1.307167
25,0.863985
30,0.684483


TrainOutput(global_step=30, training_loss=1.5285302956899007, metrics={'train_runtime': 116.866, 'train_samples_per_second': 0.428, 'train_steps_per_second': 0.257, 'total_flos': 3266150400000.0, 'train_loss': 1.5285302956899007, 'epoch': 10.0})

In [11]:
print("\n=== AFTER FINE-TUNING ===")

for p in recipe_prompts:
    output = generate_text(model2, tokenizer2, p)
    print(p)
    print("Before:", baseline2[p][:100])
    print("After :", output[:100])
    print()


=== AFTER FINE-TUNING ===
To make butter chicken
Before: To make butter chicken, cook 2 pieces and mix in the butter. When the butter is softened, add the re
After : To make butter chicken marinate chicken in yogurt and spices. Season chicken in yogurt and spices. R

For pasta carbonara
Before: For pasta carbonara, try this dish:

Pavlovioli. It is made with fresh mozzarella, garlic, basil, or
After : For pasta carbonara, cook onions until golden brown. Add spices and cook for 15 minutes. Stir in tom

To prepare chocolate cake
Before: To prepare chocolate cake for your kids, make sure you're using the right amount of sugar, and keep 
After : To prepare chocolate cake and cook until thick. Add butter and cook until thick. Stir in yogurt and 

